# Generate output and evaluate models on held-out test set

## Set Up

In [ ]:
import os
import sys

sys.path.append(os.path.abspath("../../"))
from src.config import BASE_PATH, SEED
from src.utils import (
    BEST_PARAM_DICT,
    export_data,
)
import pandas as pd
from src.eval_test import get_full_reduced_scores, aggregate_result_lists
from src.sig import prep_train_test_model

In [ ]:
train_data_dir = BASE_PATH / "data"
test_data_dir = BASE_PATH / "data/test_data_toy"
model_dir = BASE_PATH / "models"
SHARED_DICT = {
    "max_iter": 50000,
    "seed": SEED,
    "linear_type": "ridge",
}
N_BOOTS = 1000

In [ ]:
single_scores = []
paired_scores = []
for residual_setup, sub_dict in BEST_PARAM_DICT.items():
    for impact_type, param_dict in sub_dict.items():
        ## Create and train model
        X_train, y_train, X_test, y_test, model = prep_train_test_model(
            selected_params=param_dict,
            impact_type=impact_type,
            train_data_dir=train_data_dir,
            test_data_dir=test_data_dir,
            shared_dict=SHARED_DICT,
        )
        model.fit(X_train, y_train)
        export_data(
            data_to_export=model,
            export_path=model_dir / f"{residual_setup}_{impact_type}.joblib",
            remove_existing=True,
        )
        ## GET PREDICTIONS/SCORES
        # Train
        y_pred_train_df = model.predict_all(X_train)
        train_score_dict = get_full_reduced_scores(
            residual_setup=residual_setup,
            pred_df=y_pred_train_df,
            y_true=y_train,
            n_bootstraps=N_BOOTS,
            show_progress=False,
            seed=SEED,
        )
        # Test
        y_pred_test_df = model.predict_all(X_test)
        test_score_dict = get_full_reduced_scores(
            residual_setup=residual_setup,
            pred_df=y_pred_test_df,
            y_true=y_test,
            n_bootstraps=N_BOOTS,
            show_progress=False,
            seed=SEED,
        )
        single_scores, paired_scores = aggregate_result_lists(
            single_scores,
            paired_scores,
            train_score_dict,
            test_score_dict,
            residual_setup,
            impact_type,
        )


single_result_df = pd.DataFrame(single_scores)
paired_result_df = pd.DataFrame(paired_scores)